In [3]:
!pip install pypdf boto3 pyyaml

In [4]:
import boto3
from pypdf import PdfReader
from pathlib import Path
import tempfile

#### S3 settings

In [5]:
BUCKET_NAME = "multiomic-vae-literature-rag-123223178042-eu-north-1-an"

RAW_PAPERS_PREFIX = "papers/raw/"
TEXT_OUTPUT_PREFIX = "papers/text/"

s3 = boto3.client("s3")

#### list PDFs in S3

In [8]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=RAW_PAPERS_PREFIX
)

pdf_files = []

for obj in response.get("Contents", []):
    key = obj["Key"]
    
    if key.lower().endswith(".pdf"):
        pdf_files.append({
            "key": key,
            "filename": Path(key).name
        })

print(f"Found {len(pdf_files)} PDF files.")

pdf_files

Found 34 PDF files.


[{'key': 'papers/raw/111.pdf', 'filename': '111.pdf'},
 {'key': 'papers/raw/2026.02.10.705093v1.full.pdf',
  'filename': '2026.02.10.705093v1.full.pdf'},
 {'key': 'papers/raw/BindVAE.pdf', 'filename': 'BindVAE.pdf'},
 {'key': 'papers/raw/CASTLE.pdf', 'filename': 'CASTLE.pdf'},
 {'key': 'papers/raw/CAVACHON.pdf', 'filename': 'CAVACHON.pdf'},
 {'key': 'papers/raw/GNODEVAE.pdf', 'filename': 'GNODEVAE.pdf'},
 {'key': 'papers/raw/GenKI.pdf', 'filename': 'GenKI.pdf'},
 {'key': 'papers/raw/JAMIE.pdf', 'filename': 'JAMIE.pdf'},
 {'key': 'papers/raw/LiVAE.pdf', 'filename': 'LiVAE.pdf'},
 {'key': 'papers/raw/MultiVI.pdf', 'filename': 'MultiVI.pdf'},
 {'key': 'papers/raw/PIIS1097276518305471.pdf',
  'filename': 'PIIS1097276518305471.pdf'},
 {'key': 'papers/raw/Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.pdf',
  'filename': 'Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Acti

#### Extract and Save Text for All Papers

In [9]:
for paper in pdf_files:
    pdf_key = paper["key"]
    pdf_filename = paper["filename"]

    print(f"Processing: {pdf_filename}")

    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            local_pdf_path = Path(tmpdir) / pdf_filename

            # Download the PDF temporarily from S3
            s3.download_file(BUCKET_NAME, pdf_key, str(local_pdf_path))

            # Read PDF
            reader = PdfReader(str(local_pdf_path))

            text_pages = []

            for page_number, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                text_pages.append(
                    f"\n\n--- Page {page_number} ---\n\n{page_text}"
                )

            extracted_text = "\n".join(text_pages)

        # Save extracted text back to S3
        output_filename = Path(pdf_filename).stem + ".txt"
        output_key = TEXT_OUTPUT_PREFIX + output_filename

        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=output_key,
            Body=extracted_text.encode("utf-8"),
            ContentType="text/plain"
        )

        print(f"Saved: {output_key}")

    except Exception as e:
        print(f"Failed: {pdf_filename}")
        print(f"Error: {e}")

Processing: 111.pdf
Saved: papers/text/111.txt
Processing: 2026.02.10.705093v1.full.pdf
Saved: papers/text/2026.02.10.705093v1.full.txt
Processing: BindVAE.pdf
Saved: papers/text/BindVAE.txt
Processing: CASTLE.pdf
Saved: papers/text/CASTLE.txt
Processing: CAVACHON.pdf
Saved: papers/text/CAVACHON.txt
Processing: GNODEVAE.pdf
Saved: papers/text/GNODEVAE.txt
Processing: GenKI.pdf
Saved: papers/text/GenKI.txt
Processing: JAMIE.pdf
Saved: papers/text/JAMIE.txt
Processing: LiVAE.pdf
Saved: papers/text/LiVAE.txt
Processing: MultiVI.pdf
Saved: papers/text/MultiVI.txt
Processing: PIIS1097276518305471.pdf
Saved: papers/text/PIIS1097276518305471.txt
Processing: Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.pdf
Saved: papers/text/Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.txt
Processing: SCA.pdf
Saved: papers/text/SCA.txt
Processing: UnionCom.pd

#### Verify Extracted Text Files in S3

In [7]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=TEXT_OUTPUT_PREFIX
)

text_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].lower().endswith(".txt")
]

print(f"Found {len(text_files)} extracted text files.")

text_files

Found 34 extracted text files.


['papers/text/BindVAE.txt',
 'papers/text/CASTLE.txt',
 'papers/text/CAVACHON.txt',
 'papers/text/Chromatin_GeneRegulation_Review.txt',
 'papers/text/Cicero.txt',
 'papers/text/GNODEVAE.txt',
 'papers/text/GenKI.txt',
 'papers/text/JAMIE.txt',
 'papers/text/LiVAE.txt',
 'papers/text/MultiVI.txt',
 'papers/text/Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.txt',
 'papers/text/SCA.txt',
 'papers/text/UnionCom.txt',
 'papers/text/VAE_BatchCorrection_scRNAseq_Benchmark.txt',
 'papers/text/biVI.txt',
 'papers/text/cobolt.txt',
 'papers/text/factVAE.txt',
 'papers/text/hybridVI.txt',
 'papers/text/pair.txt',
 'papers/text/peakVI.txt',
 'papers/text/phd-ConvNet-VAE.txt',
 'papers/text/phd-scPair.txt',
 'papers/text/review1.txt',
 'papers/text/review2.txt',
 'papers/text/salirex.txt',
 'papers/text/scAMACE.txt',
 'papers/text/scButterfly.txt',
 'papers/text/scJVAE.txt',
 'papers/text/scMVAE.txt',
 'papers/text/scVAE.tx

#### Verify extraction quality

In [8]:
import random

sample_key = random.choice(text_files)

obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=sample_key
)

sample_text = obj["Body"].read().decode("utf-8")

print(sample_key)
print(len(sample_text))
print(sample_text[:3000])

papers/text/hybridVI.txt
124574


--- Page 1 ---

 
Degree Project in Computer Science and Engineering    Second Cycle, 30 credits  Hybrid Variational Autoencoder for Clustering of Single-Cell RNA-seq Data Introducing HybridVI, a Variational Autoencoder with two Latent Spaces 
SARAH NARROWE DANIELSSON   
Stockholm, Sweden 2022 


--- Page 2 ---

Hybrid Variational
Autoencoder for Clustering of
Single-Cell RNA-seq Data
Introducing HybridVI, a Variational
Autoencoder with two Latent Spaces
SARAH NARROWE DANIELSSON
Degree Programme in Computer Science and Engineering
Date: September 30, 2022
Supervisors: Johan Henriksson, Arvind Kumar
Examiner: Erik Fransén
School of Electrical Engineering and Computer Science
Host organization: Umeå University
Swedish title: Hybrid Variational autoencoder för analys av
enkelcells RNA-sekvensering data


--- Page 3 ---

© 2022 Sarah Narrowe Danielsson


--- Page 4 ---

Abstract | i
Abstract
Single-cell analysis means to analyze cells on an individual leve